# Byte-Pair Encoding
### *The Hundred-Page Language Models Book*, Andriy Burkov
### Section 2.3
Byte-Pair Encoding (BPE) is a tokenization algorithm that addresses the challenge of out-of-vocabulary works by breaking words into smaller units called subwords (tokens).

<b>Basic algorithm:</b>

Initialisation: Start with an initial corpus of text. Split each word in the corpus into individual characters. The initial vocabulary consists of all unique characters in the corpus. 

Iterative merging: Count all adjacent symbol pairs. Select the most frequent symbol pair. Create a new single merged symbol consisting of this pair, and replace all occurrences of the pair in the corpus by this symbol. Update the vocabulary by adding this new merged symbol.

Repeat the iterative merging until the vocabulary reaches some desired size.

In [1]:
from collections import defaultdict

In [2]:
def initialize_vocabulary(corpus, verbose=False):
    """
    generates a vocabulary that represents words as sequences of characters, and track their counts.
    Given a corpus (list of words), this function returns:
    vocabulary : a dictionary mapping each word (tokenized with spaces between characters) to its count
    charset : a set of all unique characters in the corpus
    """
    vocabulary=defaultdict(int)
    charset=set()

    if verbose:
        print(type(vocabulary))
        print(type(charset))
      
    for word in corpus:
        word_with_marker="_" + word # add "_" to the start of each word, to distinguish subwords at the beginning of words from internal subwords
        characters=list(word_with_marker) # split into characters
        charset.update(characters) # add into charset any new characters
        tokenized_word = ' '.join(characters) # join characters with spaces
        vocabulary[tokenized_word] +=1 # add to the vocabulary with count incremented by 1
      
    return vocabulary, charset

In [3]:
# define our corpus

corpus = [
    "movies", "are", "fun", "for", "everyone",
        "watching", "movies", "is", "great", "fun",
        "enjoy", "a", "great", "movie", "today",
        "research", "is","interesting", "and", "important",
        "learning", "maths", "is", "very", "important",
        "science", "discovery", "is","interesting",
        "rock ", "is", "great", "to", "listen", "to",
        "listen", "to", "music", "for", "fun",
        "music", "is","fun", "for", "everyone",
        "listen", "to", "folk", "music"
        ]

#corpus = "aabbab"

print(type(corpus))
print(len(corpus))

<class 'list'>
49


In [4]:
v,c = initialize_vocabulary(corpus)
print(type(v))
print(type(c))

print(v.items())
print(c)

<class 'collections.defaultdict'>
<class 'set'>
dict_items([('_ m o v i e s', 2), ('_ a r e', 1), ('_ f u n', 4), ('_ f o r', 3), ('_ e v e r y o n e', 2), ('_ w a t c h i n g', 1), ('_ i s', 6), ('_ g r e a t', 3), ('_ e n j o y', 1), ('_ a', 1), ('_ m o v i e', 1), ('_ t o d a y', 1), ('_ r e s e a r c h', 1), ('_ i n t e r e s t i n g', 2), ('_ a n d', 1), ('_ i m p o r t a n t', 2), ('_ l e a r n i n g', 1), ('_ m a t h s', 1), ('_ v e r y', 1), ('_ s c i e n c e', 1), ('_ d i s c o v e r y', 1), ('_ r o c k  ', 1), ('_ t o', 4), ('_ l i s t e n', 3), ('_ m u s i c', 3), ('_ f o l k', 1)])
{'f', 'e', 'v', 'r', ' ', 'u', 'w', 'h', 'n', 't', 'k', 'a', 'y', '_', 'm', 'd', 'p', 'c', 'j', 'o', 's', 'l', 'i', 'g'}


In [8]:
import re

def get_pair_counts(vocabulary):
    
  pair_counts=defaultdict(int)
    
  for tokenized_word, count in vocabulary.items():
    tokens=tokenized_word.split() # split into tokens
    for i in range(len(tokens)-1):
      pair=(tokens[i],tokens[i+1]) # form adjacent token pairs
      pair_counts[pair]+= count # add to the dictionary and increment the count
        
  return pair_counts

def merge_pair(vocabulary, pair, verbose=False):
    
  new_vocabulary={}
  bigram=re.escape(' '.join(pair)) # add backslash to special characters so they don't break everything
  pattern=re.compile(r"(?<!\S)" + bigram + r"(?!\S)") # match whole token pairs (including whitespace)
  if verbose:
    print("pair = ", pair)
    print(type(pattern))
  for tokenized_word, count in vocabulary.items():
    if verbose:
      print(tokenized_word, count)
    joined_pair = "".join(pair)
    if verbose:
      print("joined_pair = ", joined_pair)
    new_tokenized_word=pattern.sub(joined_pair, tokenized_word) # replace all occurrences of the matched pattern with the joined pair
    #new_tokenized_word=pattern.sub("".join(pair), tokenized_word) # replace all occurrences of the matched pattern with the joined pair
    new_vocabulary[new_tokenized_word]=count
  return new_vocabulary

In [10]:
def byte_pair_encoding(corpus, vocab_size, verbose : bool = False):
    """
    returns:
    vocabulary : the updated vocabulary
    merges : the list of merge operations
    charset : the original character set
    tokens : the final set of unique tokens
    """
    vocabulary, charset = initialize_vocabulary(corpus)
    merges=[]
    tokens=set(charset)
    print("initial number of tokens = ", len(tokens))
    while len(tokens) < vocab_size: # continue until the vocabulary size reaches some upper limit
        pair_counts=get_pair_counts(vocabulary)
        if not pair_counts: # if there are no more valid pairs, then exit
          break
        most_frequent_pair=max(pair_counts, key=pair_counts.get) # find the most frequent pair
        if verbose:
            print("most_frequent_pair = ", most_frequent_pair)
        merges.append(most_frequent_pair)
        vocabulary=merge_pair(vocabulary,most_frequent_pair) # merge the most frequent pair throughout the vocabulary
        #new_token=' '.join(most_frequent_pair) # create a new token from the most frequent pair
        new_token=''.join(most_frequent_pair) # create a new token from the most frequent pair
        if verbose:
            print("adding new_token = ", new_token)
        tokens.add(new_token) # add the new token to the set of all tokens

    return vocabulary, merges, charset, tokens

In [11]:
v, m, c, t = byte_pair_encoding(corpus, 50)

initial number of tokens =  24


In [12]:
print(v)

{'_movie s': 2, '_a re': 1, '_fun': 4, '_for': 3, '_ e very o n e': 2, '_ w at c h ing': 1, '_is': 6, '_ g re at': 3, '_ en j o y': 1, '_a': 1, '_movie': 1, '_to d a y': 1, '_ re s e a r c h': 1, '_i nt e re st ing': 2, '_a n d': 1, '_i m p or t a nt': 2, '_l e a r n ing': 1, '_m at h s': 1, '_ very': 1, '_ s c i en c e': 1, '_ d i s c ov ery': 1, '_ r o c k  ': 1, '_to': 4, '_l i st en': 3, '_m u s i c': 3, '_f o l k': 1}


In [13]:
print(m)

[('_', 'i'), ('_', 'f'), ('_', 'm'), ('r', 'e'), ('_i', 's'), ('o', 'r'), ('a', 't'), ('e', 'n'), ('_', 't'), ('_t', 'o'), ('s', 't'), ('o', 'v'), ('_f', 'u'), ('_fu', 'n'), ('e', 'r'), ('er', 'y'), ('i', 'n'), ('in', 'g'), ('n', 't'), ('_', 'l'), ('_m', 'ov'), ('_mov', 'i'), ('_movi', 'e'), ('_', 'a'), ('_f', 'or'), ('v', 'ery')]


In [14]:
print(c)

{'f', 'e', 'v', 'r', ' ', 'u', 'w', 'h', 'n', 't', 'k', 'a', 'y', '_', 'm', 'd', 'p', 'c', 'j', 'o', 's', 'l', 'i', 'g'}


In [15]:
print(t)

{'st', 'er', '_mov', 'v', '_a', ' ', 'u', 'n', '_fun', '_movie', '_for', 't', 'y', 'in', '_', 'm', 'p', '_m', 'o', '_is', 're', '_to', 'f', '_i', 'e', '_f', '_fu', 'ing', 'very', 'r', 'ery', '_movi', 'w', 'h', 'nt', 'ov', 'at', '_l', '_t', 'k', 'a', 'd', 'or', 'c', 'j', 's', 'l', 'i', 'g', 'en'}
